# Yu 2008 tan-loss QSPR reproduction with live AMS/DFTB descriptors

This notebook is intentionally narrower than the earlier screening tutorial. It follows Yu et al. 2008 as closely as practical while using AMS/PLAMS and DFTB:

1. load the Yu observation table and monomer/repeat proxy CSVs,
2. compute only the Yu quantum descriptor family with live DFTB calculations,
3. reproduce the paper-style train/test split and metrics,
4. fit a strict Yu-style linear and ANN baseline,
5. then add a separate refinement section using the same descriptors and, finally, RDKit descriptors.

No previous descriptor tables or result folders are used.

## Paper Target

Primary source:

- X. Yu, B. Yi, F. Liu, X. Wang, "Prediction of the dielectric dissipation factor tan d of polymers with an ANN model based on the DFT calculation", *Reactive & Functional Polymers* 68, 1557-1562, 2008. DOI: `10.1016/j.reactfunctpolym.2008.08.009`.

Yu et al. used:

- 92 observations, with the PVA 100 Hz row removed for fitting,
- 70 training rows and 21 test rows,
- frequency of measurement `v`,
- `qR+`: most positive H charge in the repeat unit,
- `q-(R/M)`: most negative repeat charge divided by most negative monomer charge,
- `EMLUMO`: monomer LUMO energy,
- `EM/RLUMO`: monomer LUMO divided by repeat-unit LUMO,
- `SR`: repeat-unit entropy,
- BP ANN architecture `[6-2-1]`.

Their reported ANN metrics are approximately:

- training: RMSE `0.01067`, Pearson `R = 0.939`,
- test: RMSE `0.01463`, Pearson `R = 0.902`.

Here the descriptors are not Gaussian03/B3LYP/6-31G(d); they are AMS DFTB3/3ob-freq descriptors. The goal is to see how close the same descriptor idea gets, then what minimal refinements help.

In [ ]:
from __future__ import annotations

import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("yu2008_dftb_reproduction_data")
OUTPUT = Path("yu2008_dftb_reproduction_outputs")
OUTPUT.mkdir(exist_ok=True)

OBS_CSV = DATA / "yu2008_tanloss_observations.csv"
MATERIALS_CSV = DATA / "yu2008_materials.csv"

observations = pd.read_csv(OBS_CSV)
materials = pd.read_csv(MATERIALS_CSV)

RUN_DFTB = True

print("Observations:", len(observations), OBS_CSV)
print("Species definitions:", len(materials), MATERIALS_CSV)

## Input Structures

Yu used monomer and repeating-unit structures. We use the same concept with small SMILES proxies:

- monomer proxies for `EMLUMO`,
- capped repeat proxies for `qR+`, `q-(R/M)`, `ERLUMO`, and `SR`.

In [ ]:
species_specs = (
    materials[materials["species_role"].isin(["monomer", "repeat"])]
    [["material_id", "polymer_key", "species_role", "oligomer_smiles", "notes"]]
    .rename(columns={"oligomer_smiles": "proxy_smiles"})
    .drop_duplicates(["polymer_key", "species_role"])
    .sort_values(["polymer_key", "species_role"])
    .reset_index(drop=True)
)

fit_rows = observations[observations["excluded_by_yu2008"] != "yes"].copy()
fit_rows["log10_frequency_Hz"] = np.log10(fit_rows["frequency_Hz"].astype(float))

print("DFTB species jobs:", len(species_specs))
print(fit_rows.groupby("split").size())
species_specs.head(8)

## Quick Monomer Views

Before running the descriptor jobs, it is useful to visually inspect a few monomer proxies. `plams.view()` will open the configured AMS/PLAMS molecule viewer in environments that support it; in a headless notebook run this cell can be skipped without affecting the QSPR workflow.

In [ ]:
import scm.plams as plams

RUN_PLAMS_VIEW = False
view_monomers = ["PVC", "PMMA", "PNVC", "PS", "PAN"]
view_specs = species_specs[
    species_specs["species_role"].eq("monomer")
    & species_specs["polymer_key"].isin(view_monomers)
].copy()

display(view_specs[["polymer_key", "material_id", "proxy_smiles", "notes"]])

for _, spec in view_specs.iterrows():
    print(f"Viewing {spec['polymer_key']} monomer: {spec['proxy_smiles']}")
    mol = plams.from_smiles(spec["proxy_smiles"])
    if RUN_PLAMS_VIEW:
        plams.view(mol)
    else:
        print("  Set RUN_PLAMS_VIEW = True in this cell to call plams.view(mol).")

## Live DFTB Descriptor Calculation

This cell uses PLAMS idioms:

- `from_smiles` to build proxy molecules,
- `Settings` and `AMSJob` for DFTB jobs,
- `AMSResults.get_charges`, `get_lumo_energies`, `get_frequencies`, and `get_zero_point_energy`,
- `AMSResults.readrkf` for the molecular thermodynamics fields,
- `Units.convert` for unit conversion.

Only the quantities needed for the Yu descriptor set are used downstream.

In [ ]:
def run_dftb_species_descriptors(specs: pd.DataFrame) -> pd.DataFrame:
    from scm.plams import AMSJob, Settings, Units, finish, from_smiles, init

    def first_scalar(value):
        arr = np.asarray(value, dtype=float).ravel()
        return float(arr[0]) if arr.size else np.nan

    def entropy_j_mol_k(results) -> float:
        try:
            entropy_au_per_k = first_scalar(results.readrkf("Thermodynamics", "Entropy total", file="dftb"))
            return float(Units.convert(entropy_au_per_k, "hartree", "kJ/mol") * 1000.0)
        except Exception:
            return np.nan

    def make_settings() -> Settings:
        s = Settings()
        s.input.ams.Task = "GeometryOptimization"
        s.input.ams.GeometryOptimization.MaxIterations = 120
        s.input.ams.Properties.NormalModes = "Yes"
        s.input.DFTB.Model = "DFTB3"
        s.input.DFTB.ResourcesDir = "DFTB.org/3ob-freq-1-2"
        s.runscript.nproc = 1
        s.runscript.preamble_lines = ["export OMP_NUM_THREADS=1"]
        return s

    def extract(results, mol) -> dict[str, float | int | bool]:
        symbols = [atom.symbol for atom in mol]
        row: dict[str, float | int | bool] = {"dftb_ok": bool(results.ok())}
        try:
            charges = np.asarray(results.get_charges(engine="dftb"), dtype=float).ravel()
        except Exception:
            charges = np.array([], dtype=float)
        h_charges = [q for q, sym in zip(charges, symbols) if sym == "H"]
        row["qH_max_e"] = float(max(h_charges)) if h_charges else np.nan
        row["q_min_e"] = float(charges.min()) if charges.size else np.nan
        row["entropy_J_mol_K"] = entropy_j_mol_k(results)
        try:
            row["lumo_eV"] = float(np.min(results.get_lumo_energies("eV", engine="dftb")))
        except Exception:
            row["lumo_eV"] = np.nan
        try:
            freqs = np.asarray(results.get_frequencies(engine="dftb"), dtype=float).ravel()
            positive = freqs[freqs > 1.0]
            row["lowest_real_frequency_cm1"] = float(positive.min()) if positive.size else np.nan
            row["n_imaginary_modes"] = int((freqs < -1.0).sum())
        except Exception:
            row["lowest_real_frequency_cm1"] = np.nan
            row["n_imaginary_modes"] = -1
        return row

    timestamp = time.strftime("%Y%m%d_%H%M%S")
    workdir_root = OUTPUT / "dftb_workdirs"
    workdir_root.mkdir(parents=True, exist_ok=True)
    init(path=str(workdir_root), folder=f"jobs_{timestamp}")
    rows = []
    try:
        for _, spec in specs.iterrows():
            mol = from_smiles(spec["proxy_smiles"])
            job = AMSJob(
                name=f"{spec['material_id']}_dftb_yu",
                molecule=mol,
                settings=make_settings(),
            )
            results = job.run()
            row = {
                "material_id": spec["material_id"],
                "polymer_key": spec["polymer_key"],
                "species_role": spec["species_role"],
                "proxy_smiles": spec["proxy_smiles"],
            }
            row.update(extract(results, mol))
            rows.append(row)
    finally:
        finish()
    return pd.DataFrame(rows)


if not RUN_DFTB:
    raise RuntimeError("This reproduction notebook is configured to compute DFTB descriptors live.")

species_dftb = run_dftb_species_descriptors(species_specs)
species_dftb.to_csv(OUTPUT / "dftb_species_descriptors.csv", index=False)
species_dftb.head(10)

## Assemble the Six Yu Inputs

The descriptors below are the DFTB analogues of the Yu paper inputs. The model section first uses exactly these descriptors plus the paper's train/test split.

In [ ]:
def assemble_yu_descriptor_table(species_df: pd.DataFrame) -> pd.DataFrame:
    mon = species_df[species_df["species_role"] == "monomer"].set_index("polymer_key")
    rep = species_df[species_df["species_role"] == "repeat"].set_index("polymer_key")
    rows = []
    for polymer_key in sorted(set(mon.index) & set(rep.index)):
        m = mon.loc[polymer_key]
        r = rep.loc[polymer_key]
        em = float(m["lumo_eV"])
        er = float(r["lumo_eV"])
        qm = float(m["q_min_e"])
        qr = float(r["q_min_e"])
        rows.append(
            {
                "polymer_key": polymer_key,
                "dftb_monomer_ok": bool(m["dftb_ok"]),
                "dftb_repeat_ok": bool(r["dftb_ok"]),
                "qRplus_H_max_e": float(r["qH_max_e"]),
                "qminus_R_over_M": qr / qm if qm != 0 else np.nan,
                "EM_LUMO_eV": em,
                "ER_LUMO_eV": er,
                "EM_over_ER_LUMO": em / er if er != 0 else np.nan,
                "SR_entropy_J_mol_K": float(r["entropy_J_mol_K"]),
                "repeat_lowest_real_frequency_cm1": float(r["lowest_real_frequency_cm1"]),
                "repeat_n_imaginary_modes": int(r["n_imaginary_modes"]),
            }
        )
    return pd.DataFrame(rows)


yu_dftb = assemble_yu_descriptor_table(species_dftb)
yu_dftb.to_csv(OUTPUT / "yu_dftb_descriptors.csv", index=False)

model_df = fit_rows.merge(yu_dftb, on="polymer_key", how="left")
model_df.to_csv(OUTPUT / "yu_dftb_model_table.csv", index=False)

yu_features_raw = [
    "frequency_Hz",
    "qRplus_H_max_e",
    "qminus_R_over_M",
    "EM_LUMO_eV",
    "EM_over_ER_LUMO",
    "SR_entropy_J_mol_K",
]
yu_features_logf = ["log10_frequency_Hz"] + yu_features_raw[1:]

print("Model rows:", len(model_df))
print("All DFTB jobs OK:", bool(species_dftb["dftb_ok"].all()))
model_df[["yu2008_no", "split", "polymer_key", "tan_delta_exp"] + yu_features_raw].head()

## Paper-style Metrics and Strict Yu Baselines

Yu reports RMSE and Pearson correlation coefficient `R` on the raw `tan(delta)` scale. We report those and scikit-learn `R2`.

The first two rows below are intentionally strict:

- MLRA: linear regression with raw measurement frequency,
- ANN `[6-2-1]`: a small two-hidden-node neural net with raw measurement frequency.

These are the cleanest DFTB analogues of the paper setup, but the DFTB descriptors are not numerically identical to B3LYP/6-31G(d), so exact reproduction is not expected.

In [ ]:
from sklearn.base import clone
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR


train_mask = model_df["split"].to_numpy() == "train"
test_mask = model_df["split"].to_numpy() == "test"
y = model_df["tan_delta_exp"].astype(float).to_numpy()


def pearson_r(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if np.std(y_pred) < 1e-14 or np.std(y_true) < 1e-14:
        return np.nan
    return float(np.corrcoef(y_true, y_pred)[0, 1])


def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def evaluate_named_model(name: str, model, feature_cols: list[str]) -> tuple[dict, np.ndarray]:
    work = model_df.dropna(subset=feature_cols + ["tan_delta_exp"]).copy()
    tr = work["split"].to_numpy() == "train"
    te = work["split"].to_numpy() == "test"
    X = work[feature_cols].astype(float).to_numpy()
    yy = work["tan_delta_exp"].astype(float).to_numpy()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        fitted = clone(model)
        fitted.fit(X[tr], yy[tr])
        pred = np.clip(fitted.predict(X), 0, None)

    def block(mask):
        return {
            "rmse": rmse(yy[mask], pred[mask]),
            "mae": float(mean_absolute_error(yy[mask], pred[mask])),
            "R": pearson_r(yy[mask], pred[mask]),
            "R2": float(r2_score(yy[mask], pred[mask])),
        }

    row = {
        "model": name,
        "features": ",".join(feature_cols),
        "n_train": int(tr.sum()),
        "n_test": int(te.sum()),
    }
    for prefix, vals in [("train", block(tr)), ("test", block(te))]:
        row.update({f"{prefix}_{k}": v for k, v in vals.items()})
    pred_table = work[["yu2008_no", "split", "polymer_key", "frequency_Hz", "tan_delta_exp"]].copy()
    pred_table[f"pred_{name}"] = pred
    return row, pred_table


models_to_compare = [
    (
        "Yu_DFTB_MLRA_raw_frequency",
        make_pipeline(StandardScaler(), LinearRegression()),
        yu_features_raw,
    ),
    (
        "Yu_DFTB_ANN_6_2_1_raw_frequency",
        make_pipeline(
            StandardScaler(),
            MLPRegressor(hidden_layer_sizes=(2,), activation="logistic", solver="lbfgs", alpha=1e-7, max_iter=8000, random_state=22),
        ),
        yu_features_raw,
    ),
]

rows = []
prediction_tables = []
for name, model, features in models_to_compare:
    row, preds = evaluate_named_model(name, model, features)
    rows.append(row)
    prediction_tables.append(preds)

strict_results = pd.DataFrame(rows)
strict_results

## Minimal Refinement: Same Yu Descriptors, Better Frequency Scaling/Model

This section still uses only the six Yu inputs, but with `log10(frequency)` instead of raw frequency. That is a modest modernization: dielectric loss is usually discussed across logarithmic frequency decades, and the Yu data span 60 Hz to 10 MHz.

The nonlinear refinements below are not claimed to reproduce Yu's exact BP ANN. They answer a practical question: with DFTB descriptors, how close can we get while keeping the descriptor set unchanged?

In [ ]:
refinement_models = [
    (
        "Yu_DFTB_ANN_log_frequency",
        make_pipeline(
            StandardScaler(),
            MLPRegressor(hidden_layer_sizes=(6,), activation="tanh", solver="lbfgs", alpha=1e-4, max_iter=8000, random_state=72),
        ),
        yu_features_logf,
    ),
    (
        "Yu_DFTB_SVR_log_frequency",
        make_pipeline(StandardScaler(), SVR(C=1000.0, gamma=0.01, epsilon=1e-4)),
        yu_features_logf,
    ),
    (
        "Yu_DFTB_KRR_log_frequency",
        make_pipeline(StandardScaler(), KernelRidge(alpha=1e-5, kernel="rbf", gamma=0.01)),
        yu_features_logf,
    ),
]

rows = []
for name, model, features in refinement_models:
    row, preds = evaluate_named_model(name, model, features)
    rows.append(row)
    prediction_tables.append(preds)

yu_only_refinement = pd.DataFrame(rows)
yu_only_refinement

## Descriptor Refinement: Add RDKit Repeat Descriptors

This is no longer a Yu-only model. It is included as a short refinement path after the Yu reproduction attempt.

RDKit descriptors can encode repeat-proxy composition/topology cheaply. They may improve interpolation in this small table, but grouped or external validation would be needed before using them for new chemistry.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Crippen, Descriptors, Lipinski, rdMolDescriptors
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor


def rdkit_descriptors(smiles: str) -> dict[str, float]:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(smiles)
    mol_h = Chem.AddHs(mol)
    atoms = list(mol_h.GetAtoms())
    total_atoms = max(len(atoms), 1)
    heavy_atoms = max(mol.GetNumHeavyAtoms(), 1)
    counts = {el: sum(1 for atom in atoms if atom.GetSymbol() == el) for el in ["C", "H", "N", "O", "F", "Cl", "Br", "I"]}
    aromatic_atoms = sum(1 for atom in mol.GetAtoms() if atom.GetIsAromatic())
    return {
        "rdkit_MolWt": float(Descriptors.MolWt(mol)),
        "rdkit_TPSA": float(rdMolDescriptors.CalcTPSA(mol)),
        "rdkit_MolLogP": float(Crippen.MolLogP(mol)),
        "rdkit_HBA": float(Lipinski.NumHAcceptors(mol)),
        "rdkit_HBD": float(Lipinski.NumHDonors(mol)),
        "rdkit_RotB": float(Lipinski.NumRotatableBonds(mol)),
        "rdkit_AromaticRings": float(rdMolDescriptors.CalcNumAromaticRings(mol)),
        "rdkit_HeteroFrac": float(sum(v for k, v in counts.items() if k not in {"C", "H"}) / total_atoms),
        "rdkit_HalogenFrac": float(sum(counts[k] for k in ["F", "Cl", "Br", "I"]) / total_atoms),
        "rdkit_ONFrac": float((counts["O"] + counts["N"]) / total_atoms),
        "rdkit_AromaticFrac": float(aromatic_atoms / heavy_atoms),
    }


repeat_smiles = (
    species_specs[species_specs["species_role"] == "repeat"]
    [["polymer_key", "proxy_smiles"]]
    .drop_duplicates("polymer_key")
)
rdkit_rows = []
for _, row in repeat_smiles.iterrows():
    item = {"polymer_key": row["polymer_key"]}
    item.update(rdkit_descriptors(row["proxy_smiles"]))
    rdkit_rows.append(item)
rdkit_df = pd.DataFrame(rdkit_rows)
rdkit_cols = [c for c in rdkit_df.columns if c != "polymer_key"]

model_df = model_df.merge(rdkit_df, on="polymer_key", how="left")
rdkit_log_features = ["log10_frequency_Hz"] + rdkit_cols
yu_rdkit_log_features = yu_features_logf + rdkit_cols

rdkit_models = [
    ("RDKit_SVR_log_frequency", make_pipeline(StandardScaler(), SVR(C=1000.0, gamma=0.01, epsilon=1e-4)), rdkit_log_features),
    ("RDKit_KRR_log_frequency", make_pipeline(StandardScaler(), KernelRidge(alpha=1e-5, kernel="rbf", gamma=0.01)), rdkit_log_features),
    ("Yu_plus_RDKit_SVR_log_frequency", make_pipeline(StandardScaler(), SVR(C=1000.0, gamma=0.01, epsilon=1e-4)), yu_rdkit_log_features),
]

rows = []
for name, model, features in rdkit_models:
    row, preds = evaluate_named_model(name, model, features)
    rows.append(row)
    prediction_tables.append(preds)

rdkit_refinement = pd.DataFrame(rows)
rdkit_refinement

## Comparison Against Yu's Reported Metrics

The strict DFTB reproduction is expected to be worse than Yu's Gaussian/B3LYP descriptors. The important observation is whether a small, transparent refinement gets back to their reported scale.

In [ ]:
paper_reference = pd.DataFrame(
    [
        {
            "model": "Yu_2008_reported_ANN",
            "train_rmse": 0.01067,
            "train_R": 0.939,
            "test_rmse": 0.01463,
            "test_R": 0.902,
            "note": "Reported in Yu et al. 2008; Gaussian03/B3LYP/6-31G(d), BP ANN [6-2-1]",
        }
    ]
)

all_results = pd.concat([strict_results, yu_only_refinement, rdkit_refinement], ignore_index=True)
compact = all_results[
    ["model", "train_rmse", "train_R", "train_R2", "test_rmse", "test_R", "test_R2"]
].sort_values("test_rmse")
compact.to_csv(OUTPUT / "model_comparison.csv", index=False)
compact

In [ ]:
best_name = compact.iloc[0]["model"]
merged_predictions = prediction_tables[0]
for tab in prediction_tables[1:]:
    pred_col = [c for c in tab.columns if c.startswith("pred_")][0]
    merged_predictions = merged_predictions.merge(
        tab[["yu2008_no", pred_col]],
        on="yu2008_no",
        how="left",
    )
merged_predictions.to_csv(OUTPUT / "prediction_table.csv", index=False)
merged_predictions.head()

## Parity Plots

The parity plots compare experimental tan(delta) against model predictions for the paper-style train/test split. They are mainly diagnostic: the Yu split is useful for reproducing the paper, while prospective screening should also use grouped validation by polymer family.

In [ ]:
import os

os.environ.setdefault("MPLCONFIGDIR", str(OUTPUT / "mplconfig"))
(OUTPUT / "mplconfig").mkdir(exist_ok=True)

import matplotlib.pyplot as plt

plot_models = [
    "pred_Yu_DFTB_SVR_log_frequency",
    "pred_RDKit_SVR_log_frequency",
    "pred_Yu_plus_RDKit_SVR_log_frequency",
]
plot_models = [col for col in plot_models if col in merged_predictions.columns]

fig, axes = plt.subplots(1, len(plot_models), figsize=(5.2 * len(plot_models), 4.6), squeeze=False)
axes = axes.ravel()

for ax, col in zip(axes, plot_models):
    for split, marker, color in [("train", "o", "#4575b4"), ("test", "s", "#d73027")]:
        part = merged_predictions[merged_predictions["split"].eq(split)]
        ax.scatter(
            part["tan_delta_exp"],
            part[col],
            label=split,
            marker=marker,
            s=42,
            alpha=0.82,
            edgecolor="white",
            linewidth=0.5,
            color=color,
        )
    finite = merged_predictions[["tan_delta_exp", col]].replace([np.inf, -np.inf], np.nan).dropna()
    upper = float(max(finite["tan_delta_exp"].max(), finite[col].max()) * 1.08)
    ax.plot([0, upper], [0, upper], color="#333333", linewidth=1.0, linestyle="--")
    ax.set_xlim(0, upper)
    ax.set_ylim(0, upper)
    ax.set_xlabel("Experimental tan(delta)")
    ax.set_ylabel("Predicted tan(delta)")
    ax.set_title(col.replace("pred_", "").replace("_", " "), fontsize=10)
    ax.grid(True, alpha=0.25)

axes[0].legend(frameon=False)
fig.tight_layout()
parity_png = OUTPUT / "parity_plots.png"
fig.savefig(parity_png, dpi=180, bbox_inches="tight")
plt.show()

print(f"Wrote {parity_png}")

In [ ]:
summary = {
    "paper_reference": paper_reference.to_dict(orient="records")[0],
    "best_model_by_test_rmse": compact.iloc[0].to_dict(),
    "strict_yu_dftb": strict_results.to_dict(orient="records"),
    "yu_only_refinement": yu_only_refinement.to_dict(orient="records"),
    "rdkit_refinement": rdkit_refinement.to_dict(orient="records"),
}
(OUTPUT / "summary.json").write_text(json.dumps(summary, indent=2))
summary

## Next Refinement Options

Useful next steps, kept outside this strict notebook:

1. Replace DFTB orbital/charge descriptors with ADF single-points on the DFTB geometries, for example PBE, r2SCAN, a hybrid, or a range-separated hybrid.
2. Keep DFTB frequencies/entropy for speed but use DFT charges and LUMO descriptors.
3. Add COSMO-RS or sigma-profile descriptors for polarity/segment interactions.
4. Add conformer ensembles for flexible methacrylates and aromatic side chains.
5. Use modern validation discipline: the Yu train/test split for reproduction, then grouped polymer-family validation for prospective screening.